In [1]:
import pandas as pd

df = pd.read_csv("extrato_sujo.csv")
df.shape

(42, 6)

In [9]:
df.head()

,id_transacao,data,descricao,categoria,valor,natureza
0,1,2026-10-02,Salário,Salário,2850.00,Entrada
1,2,2026-10-03,Mercado,Alimentação,24.83,Saída
2,3,2026-10-04,Aplicativo de transporte,Transporte,75.16,Saída
3,4,05/10/2026,Consulta,Saúde,157.75,Saída
4,5,2026-10-06,Ônibus,Transporte,34.17,Saída


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_transacao  42 non-null     int64  
 1   data          42 non-null     str    
 2   descricao     42 non-null     str    
 3   categoria     40 non-null     str    
 4   valor         39 non-null     float64
 5   natureza      42 non-null     str    
dtypes: float64(1), int64(1), str(4)
memory usage: 2.1 KB


In [3]:
df.isnull().sum()

id_transacao    0
data            0
descricao       0
categoria       2
valor           3
natureza        0
dtype: int64

In [5]:
df.duplicated().sum()

np.int64(2)

In [6]:
df[df.duplicated(keep=False)]

,id_transacao,data,descricao,categoria,valor,natureza
4,5,2026-10-06,Ônibus,Transporte,34.17,Saída
5,5,2026-10-06,Ônibus,Transporte,34.17,Saída
23,23,2026-10-20,Curso online,Educação,64.82,Saída
24,23,2026-10-20,Curso online,Educação,64.82,Saída


In [7]:
df["categoria"].value_counts()

categoria
Alimentação    8
Transporte     6
Saúde          5
Moradia        4
Outros         4
Educação       4
Lazer          4
Salário        2
lazer          1
MORADIA        1
 Saúde         1
Name: count, dtype: int64

In [8]:
df["valor"].describe()

count       39.000000
mean       641.158974
std       3014.668018
min      -2850.000000
25%         64.820000
50%        107.920000
75%        199.575000
max      18500.000000
Name: valor, dtype: float64

problemas encontrados:
3 valores vazios
2 categorias vazias
4 datas escritas como dd/mm/aaaa no meio das outras (aaaa-mm-dd)
3 textos de categoria inconsistentes
1 valor gigante (outlier)
1 valor negativo em uma entrada
2 linhas duplicadas

In [10]:
df = df.drop_duplicates() # para apagar as linhas duplicadas

In [12]:
df["categoria"] = df["categoria"].str.strip().str.title()
df["categoria"].value_counts()
# padronizar os textos

categoria
Alimentação    8
Saúde          6
Transporte     5
Moradia        5
Lazer          5
Outros         4
Educação       3
Salário        2
Name: count, dtype: int64

In [13]:
d1 = pd.to_datetime(df["data"], format="%Y-%m-%d", errors="coerce")
d2 = pd.to_datetime(df["data"], format="%d/%m/%Y", errors="coerce")
df["data"] = d1.fillna(d2)
df["mes"] = df["data"].dt.month
# padronizar as datas

In [15]:
df["categoria"] = df["categoria"].fillna("Não informada")
df = df.dropna(subset=["valor"])
df.shape
# concertando os valores vazios, onde ha dados vitais como dinheiro a linha deve ser excluida

(37, 7)

In [16]:
df[(df["natureza"] == "Entrada") & (df["valor"] < 0)]

,id_transacao,data,descricao,categoria,valor,natureza,mes
21,21,2026-10-19,Salário,Salário,-2850.0,Entrada,10


In [17]:
df.loc[(df["natureza"] == "Entrada") & (df["valor"] < 0), "valor"] = df["valor"].abs()

In [ ]:
df = df[df["valor"] <= 5000] # retira valores maiores que 5000 
df.shape

(36, 7)

In [20]:
print(df.shape)

(36, 7)


In [21]:
print(df.isnull().sum())

id_transacao    0
data            0
descricao       0
categoria       0
valor           0
natureza        0
mes             0
dtype: int64


In [22]:
print(df.duplicated().sum())

0


In [23]:
df.info()

<class 'pandas.DataFrame'>
Index: 36 entries, 0 to 41
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id_transacao  36 non-null     int64         
 1   data          36 non-null     datetime64[us]
 2   descricao     36 non-null     str           
 3   categoria     36 non-null     str           
 4   valor         36 non-null     float64       
 5   natureza      36 non-null     str           
 6   mes           36 non-null     int32         
dtypes: datetime64[us](1), float64(1), int32(1), int64(1), str(3)
memory usage: 2.1 KB


In [24]:
saidas = df[df["natureza"] == "Saída"]
saidas.groupby("categoria")["valor"].sum().sort_values(ascending=False)

categoria
Moradia          1741.17
Não informada    1210.14
Saúde            1042.26
Educação          517.76
Lazer             475.72
Transporte        321.04
Alimentação       280.28
Name: valor, dtype: float64

In [ ]:
# agora é possivel ver e analisar os dados limpos!

In [25]:
df.to_csv("extrato_limpo.csv", index=False)